In [48]:
import pandas as pd
import numpy as np
from scipy.special import softmax

In [ ]:
from pathlib import Path

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

root = Path(r"d:\HATCH\R1")
metadata_path = root / "metadata.xlsx"
mean_path = root / "mean_of_z_score.xlsx"

metadata = pd.read_excel(metadata_path)
mean_df = pd.read_excel(mean_path)


In [50]:
mean_df[['T_SRS_AWR', 'T_SRS_COG', 'T_SRS_COMM', 'T_SRS_MOT',
       'T_SRS_RRB', 'CBCL_AP_T', 'CBCL_Externalizing_T', 'ASC_PA', 'ASC_AA',
       'ASC_SA', 'ASC_uncertainty']] = mean_df[['T_SRS_AWR', 'T_SRS_COG', 'T_SRS_COMM', 'T_SRS_MOT',
       'T_SRS_RRB', 'CBCL_AP_T', 'CBCL_Externalizing_T', 'ASC_PA', 'ASC_AA',
       'ASC_SA', 'ASC_uncertainty']].round(decimals=0)

In [51]:
mean_df

,sample_id,T_SRS_AWR,T_SRS_COG,T_SRS_COMM,T_SRS_MOT,T_SRS_RRB,CBCL_AP_T,CBCL_Externalizing_T,ASC_PA,ASC_AA,ASC_SA,ASC_uncertainty,M_SEQ_hypo,M_SEQ_hyper,M_SEQ_seeking
0,A0003,64,70,71,73,68,59,59,3,3,8,6,2.500000,3.071429,2.769231
1,A0015,76,76,69,60,64,68,66,5,2,5,6,2.333333,2.357143,2.583333
2,A0173,70,72,74,60,64,82,47,8,2,0,8,1.500000,1.428571,1.076923
3,A0215,71,77,72,71,60,73,74,7,3,4,7,1.666667,2.142857,2.153846
4,A0235,64,68,70,67,62,61,56,2,0,7,1,1.166667,1.428571,1.692308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
722,A1127,60,61,61,52,50,53,43,0,0,1,1,1.500000,1.214286,1.230769
723,A1235,64,74,69,77,73,50,58,9,2,10,17,2.166667,3.714286,1.692308
724,A1239,64,63,58,50,50,61,60,5,0,3,0,2.333333,1.571429,1.692308
725,A1240,64,79,79,79,78,66,52,7,3,4,8,1.833333,2.428571,2.461538


In [52]:
feature_cols = [col for col in mean_df.columns if col not in ["sample_id"]]
metadata_cluster = metadata[["sample_id", "Cluster"]].copy()
merged_mean = mean_df.merge(metadata_cluster, on="sample_id", how="left")

In [53]:
control_mask = metadata["Cluster"] == "Control"
control_df = metadata.loc[control_mask, ["sample_id", "Cluster", *feature_cols]].copy()

In [54]:
combined_raw = pd.concat(
    [merged_mean[["sample_id", *feature_cols, "Cluster"]], control_df],
    ignore_index=True,
)
combined_raw = combined_raw[["sample_id", "Cluster", *feature_cols]]

imputer = KNNImputer(n_neighbors=2)
combined_raw.loc[:, feature_cols] = imputer.fit_transform(combined_raw[feature_cols])

non_control_rows = combined_raw["Cluster"] != "Control"
control_rows = combined_raw["Cluster"] == "Control"

scaler = StandardScaler()
combined = combined_raw.copy()
combined.loc[non_control_rows, feature_cols] = scaler.fit_transform(
    combined_raw.loc[non_control_rows, feature_cols]
)
combined.loc[control_rows, feature_cols] = scaler.transform(
    combined_raw.loc[control_rows, feature_cols]
)

In [56]:
cluster_order = ["Control", "Mild", "Moderate", "Severe"]
cluster_scores = {label: score for score, label in enumerate(cluster_order)}
missing_clusters = sorted(set(cluster_order) - set(combined["Cluster"].dropna().unique()))

cluster_centers = (
    combined.groupby("Cluster")[feature_cols]
    .mean()
    .reindex(cluster_order)
)

control_center = cluster_centers.loc["Control"].to_numpy(dtype=float)
cluster_distances = {"Control": 0.0}
for label in cluster_order[1:]:
    group_center = cluster_centers.loc[label].to_numpy(dtype=float)
    cluster_distances[label] = float(np.linalg.norm(group_center - control_center))

feature_matrix = combined[feature_cols].to_numpy(dtype=float)
center_matrix = cluster_centers.to_numpy(dtype=float)
diff = feature_matrix[:, None, :] - center_matrix[None, :, :]
squared_distances = np.sum(diff ** 2, axis=2)
score_vector = np.array([cluster_scores[label] for label in cluster_order], dtype=float)

tau_values = [1.0]
soft_score_cols = []
for tau in tau_values:
    probabilities = softmax(-squared_distances / tau, axis=1)
    tau_label = f"{tau:.1f}"
    score_col = f"combined_tau_{tau_label}"
    combined[score_col] = probabilities @ score_vector
    soft_score_cols.append(score_col)

asd_index_path = root / "ASD_index.xlsx"
with pd.ExcelWriter(asd_index_path, engine="openpyxl") as writer:
    combined.to_excel(writer, sheet_name="soft_index", index=False)

combined[["sample_id", "Cluster", *soft_score_cols]].head()

,sample_id,Cluster,combined_tau_1.0
0,A0003,Moderate,2.045866
1,A0015,Moderate,2.001506
2,A0173,Moderate,1.996588
3,A0215,Moderate,2.005830
4,A0235,Mild,1.300200
